In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df=pd.read_csv('matches_2008-2024.csv')
df.head(4)

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2008,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2008,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2008,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2008,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper


### So here we dont need few fetures like umpire,target_overs(mainly 20),player_of_match,date


In [ ]:
df=df.drop(columns=['date','player_of_match','target_overs','method','umpire1','umpire2','season','super_over'])

In [ ]:
#lets get all unique character form object type features
columns=['match_type','venue','team1','team2','toss_winner','toss_decision','winner','result',]
for col in columns:
    print("columns->",col,df[col].unique())

In [ ]:
df.head(4)

In [ ]:
venue_mapping = {
    'Arun Jaitley Stadium, Delhi':'Arun Jaitley Stadium',
    
    'Brabourne Stadium, Mumbai':'Brabourne Stadium',
    'Wankhede stadium, mumbai': 'Wankhede stadium',
    'Dr DY Patil Sports Academy, Mumbai':'Dr DY Patil Sports Academy',
   
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam':'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',
    
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    
    'Eden Gardens, Kolkata':'Eden Gardens',
   
    'MA Chidambaram Stadium, Chepauk':'MA Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai':'MA Chidambaram Stadium',
    'M Chinnaswamy Stadium, Bengaluru':'M Chinnaswamy Stadium',
    'M.Chinnaswamy Stadium':'M Chinnaswamy Stadium',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala':'Himachal Pradesh Cricket Association Stadium',
    
    'Rajiv Gandhi International Stadium, Uppal':'Rajiv Gandhi International Stadium',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad':'Rajiv Gandhi International Stadium',
   
    'Maharashtra Cricket Association Stadium, Pune':'Maharashtra Cricket Association Stadium',
   'Sawai Mansingh Stadium, Jaipur':'Sawai Mansingh Stadium',
}

df['venue'] = df['venue'].replace(venue_mapping)

In [ ]:
unique_venues = sorted(df['venue'].unique())
print(len(unique_venues))

In [ ]:

#bangalore->banalure both same
df['city']=df['city'].replace('Bengaluru','Bangalore')



In [ ]:
df.dropna(subset=['winner'],inplace=True)

In [ ]:
df['match_type'].unique()

In [ ]:
df['team1'].unique()

In [ ]:

for x in ['team1','team2','toss_winner','winner']:
   df[x]=df[x].replace('Royal Challengers Bengaluru','Royal Challengers Bangalore')
   df[x]=df[x].replace('Rising Pune Supergiant','Rising Pune Supergiants')
   df[x]=df[x].replace('Delhi Daredevils','Delhi Capitals')
   df[x]=df[x].replace('Deccan Chargers','Sunrisers Hyderabad')
   df[x]=df[x].replace('Gujarat Lions','Gujarat Titans')
   df[x]=df[x].replace('Kings XI Punjab','Punjab Kings')



In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(df['team1'])  # fit on all team names

df['team1'] = le.transform(df['team1'])
df['team2'] = le.transform(df['team2'])
df['toss_winner'] = le.transform(df['toss_winner'])
df['winner'] = le.transform(df['winner'])

df['toss_decision'] = df['toss_decision'].map({'bat': 1, 'field': 0})


In [ ]:
df['result'] = df['result'].map({'runs': 1, 'wickets': 0})

In [ ]:
le1=LabelEncoder()
df['city']=le1.fit_transform(df['city'])

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ord_type = OrdinalEncoder(categories=[[
    'League',
    'Eliminator',
    'Qualifier 1',
    'Qualifier 2',
    'Elimination Final',
    'Semi Final',
    '3rd Place Play-Off',
    'Final'
]])
df['match_type']=ord_type.fit_transform(df[['match_type']])

In [ ]:
le2=LabelEncoder()
df['venue']=le2.fit_transform(df['venue'])

In [ ]:
df['win_by_run']=df.apply(lambda x:x['result_margin'] if x['result']==1.0 else 0,axis=1)
df['win_by_wicket']=df.apply(lambda x:x['result_margin'] if x['result']==0.0 else 0,axis=1)

In [ ]:
X=df.drop(columns=['id','winner','result_margin','win_by_run','win_by_wicket','target_runs','result'],axis=1)
y=df['winner']

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)


In [ ]:
from xgboost import XGBClassifier
# Example usage
model = XGBClassifier(n_estimators=100, learning_rate=0.1,max_depth=7,colsample_bytree=1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)



In [ ]:
from sklearn.metrics import classification_report
print("Classification report",classification_report(y_test,predictions))

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',   # use 'neg_mean_squared_error' for regression
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}
grid = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

In [ ]:
import pickle

pickle.dump(model, open('model.pkl','wb'))
pickle.dump(le, open('le.pkl','wb'))
pickle.dump(le1, open('le1.pkl','wb'))
pickle.dump(le2, open('le2.pkl','wb'))
pickle.dump(ord_type, open('ord.pkl','wb'))

In [3]:
ddf=pd.read_csv('matches_2008-2024.csv')

In [4]:
ddf=ddf.drop(columns=['date','player_of_match','target_overs','method','umpire1','umpire2','super_over'])

In [5]:
ddf.head(4)

,id,season,city,match_type,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs
0,335982,2008,Bangalore,League,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0
1,335983,2008,Chandigarh,League,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0
2,335984,2008,Delhi,League,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0
3,335985,2008,Mumbai,League,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0


In [6]:
unique_venues = sorted(ddf['venue'].unique())
unique_venues


['Arun Jaitley Stadium',
 'Arun Jaitley Stadium, Delhi',
 'Barabati Stadium',
 'Barsapara Cricket Stadium, Guwahati',
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow',
 'Brabourne Stadium',
 'Brabourne Stadium, Mumbai',
 'Buffalo Park',
 'De Beers Diamond Oval',
 'Dr DY Patil Sports Academy',
 'Dr DY Patil Sports Academy, Mumbai',
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam',
 'Dubai International Cricket Stadium',
 'Eden Gardens',
 'Eden Gardens, Kolkata',
 'Feroz Shah Kotla',
 'Green Park',
 'Himachal Pradesh Cricket Association Stadium',
 'Himachal Pradesh Cricket Association Stadium, Dharamsala',
 'Holkar Cricket Stadium',
 'JSCA International Stadium Complex',
 'Kingsmead',
 'M Chinnaswamy Stadium',
 'M Chinnaswamy Stadium, Bengaluru',
 'M.Chinnaswamy Stadium',
 'MA Chidambaram Stadium',
 'MA Chidambaram Stadium, Chepauk',
 'MA Chidambaram Stadium, Chepauk, Chennai',
 'Maharaja

In [7]:
venue_mapping = {
    'Arun Jaitley Stadium, Delhi':'Arun Jaitley Stadium',
    
    'Brabourne Stadium, Mumbai':'Brabourne Stadium',
    'Wankhede stadium, mumbai': 'Wankhede stadium',
    'Dr DY Patil Sports Academy, Mumbai':'Dr DY Patil Sports Academy',
   
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam':'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala':'Himachal Pradesh Cricket Association Stadium',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    
    'Eden Gardens, Kolkata':'Eden Gardens',
   
    'MA Chidambaram Stadium, Chepauk':'MA Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai':'MA Chidambaram Stadium',
    'M Chinnaswamy Stadium, Bengaluru':'M Chinnaswamy Stadium',
    'M.Chinnaswamy Stadium':'M Chinnaswamy Stadium',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala':'Himachal Pradesh Cricket Association Stadium',

    'Punjab Cricket Association IS Bindra Stadium, Mohali':'Punjab Cricket Association IS Bindra Stadium',
    'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh':'Punjab Cricket Association IS Bindra Stadium',
    'Punjab Cricket Association Stadium, Mohali':'Punjab Cricket Association IS Bindra Stadium',
    'Rajiv Gandhi International Stadium, Uppal':'Rajiv Gandhi International Stadium',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad':'Rajiv Gandhi International Stadium',
   
    'Maharashtra Cricket Association Stadium, Pune':'Maharashtra Cricket Association Stadium',
   'Sawai Mansingh Stadium, Jaipur':'Sawai Mansingh Stadium',
}

ddf['venue'] = ddf['venue'].replace(venue_mapping)

In [8]:
ddf['city']=ddf['city'].replace('Bengaluru','Bangalore')

In [9]:
for x in ['team1','team2','toss_winner','winner']:
   ddf[x]=ddf[x].replace('Royal Challengers Bengaluru','Royal Challengers Bangalore')
   ddf[x]=ddf[x].replace('Rising Pune Supergiant','Rising Pune Supergiants')
   ddf[x]=ddf[x].replace('Delhi Daredevils','Delhi Capitals')
   ddf[x]=ddf[x].replace('Deccan Chargers','Sunrisers Hyderabad')
   ddf[x]=ddf[x].replace('Gujarat Lions','Gujarat Titans')
   ddf[x]=ddf[x].replace('Kings XI Punjab','Punjab Kings')

In [10]:
ddf['team1'].unique()

array(['Royal Challengers Bangalore', 'Punjab Kings', 'Delhi Capitals',
       'Mumbai Indians', 'Kolkata Knight Riders', 'Rajasthan Royals',
       'Sunrisers Hyderabad', 'Chennai Super Kings',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Gujarat Titans',
       'Rising Pune Supergiants', 'Lucknow Super Giants'], dtype=object)

In [11]:
ddf['toss_decision'] = ddf['toss_decision'].map({'bat': 1, 'field': 0})

In [12]:
ddf.head(4)

,id,season,city,match_type,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs
0,335982,2008,Bangalore,League,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,0,Kolkata Knight Riders,runs,140.0,223.0
1,335983,2008,Chandigarh,League,Punjab Cricket Association IS Bindra Stadium,Punjab Kings,Chennai Super Kings,Chennai Super Kings,1,Chennai Super Kings,runs,33.0,241.0
2,335984,2008,Delhi,League,Arun Jaitley Stadium,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,1,Delhi Capitals,wickets,9.0,130.0
3,335985,2008,Mumbai,League,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,1,Royal Challengers Bangalore,wickets,5.0,166.0


In [13]:
ddf=ddf.drop(columns=['result','result_margin','target_runs'])

In [14]:
X_data=ddf.drop(columns=['id','winner'],axis=1)

In [15]:
y_data=ddf['winner']

In [16]:
nominal_col=['city','venue','team1','team2','toss_winner']
ordinal_col=['match_type']

In [17]:
from sklearn.preprocessing import LabelEncoder
la=LabelEncoder()
y_encoded_data=la.fit_transform(y_data)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
ct=ColumnTransformer(
    transformers=[
        ('nominal',OneHotEncoder(handle_unknown='ignore'),nominal_col),
        ('ord',OrdinalEncoder(categories=[[
                'League',
                'Eliminator',
                'Qualifier 1',
                'Qualifier 2',
                'Elimination Final',
                'Semi Final',
                '3rd Place Play-Off',
                'Final'
            ]]),ordinal_col),
       
    ],
    remainder='passthrough'
)




In [31]:
from xgboost import XGBClassifier
pipeline=Pipeline(
    [
        ('preprocessor',ct),
        ('classifier',XGBClassifier(n_estimators=100, learning_rate=0.1,max_depth=7,colsample_bytree=1,use_label_encoder=False)),
    ]
)

In [32]:
from sklearn.model_selection import train_test_split

X_train_new,X_test_new,y_train_new,y_test_new=train_test_split(X_data,y_encoded_data,test_size=0.3,random_state=42)

In [34]:
pipeline.fit(X_train_new,y_train_new)
y_new_pred=pipeline.predict(X_test_new)
from sklearn.metrics import classification_report
print(classification_report(y_test_new,y_new_pred))

[15:39:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.



              precision    recall  f1-score   support

           0       0.46      0.56      0.51        34
           1       0.52      0.39      0.44        36
           2       0.67      0.40      0.50        20
           3       0.00      0.00      0.00         2
           4       0.56      0.47      0.51        43
           5       0.44      0.50      0.47         8
           6       0.48      0.62      0.54        37
           7       0.25      0.50      0.33         2
           8       0.34      0.32      0.33        34
           9       0.47      0.42      0.44        38
          10       0.33      0.33      0.33         6
          11       0.55      0.52      0.53        46
          12       0.36      0.59      0.45        22
          13       0.00      0.00      0.00         1

    accuracy                           0.47       329
   macro avg       0.39      0.40      0.39       329
weighted avg       0.48      0.47      0.47       329



/Users/piyushajitchavan/Desktop/TransferMain/PYTHON/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/piyushajitchavan/Desktop/TransferMain/PYTHON/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/piyushajitchavan/Desktop/TransferMain/PYTHON/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to